# My first regional brain volumes
## Goal
Download one OpenNeuro T1w scan → keep it in Drive → segment it with FastSurfer → inspect labels → export regional volumes.

**Teaching pilot — complete rerun and supervisor validation pending.** The supplied Colab notebook contains saved outputs through regional-volume export. This public copy clears those outputs and has not been rerun end to end here. Treat errors as feedback for the course. Run cells in order using ▶; stop at any red error.

Use a Google account and preferably a computer browser. Choose **Runtime → Change runtime type → T4 GPU**, then **File → Save a copy in Drive**. GPU availability is controlled by Colab; CPU is a slower fallback. No paid plan is required by this notebook, but resources are not guaranteed.

This is a single-scan learning exercise, not a diagnostic assessment or a group study. Data: [ds004144](https://openneuro.org/datasets/ds004144), a study of emotion regulation in females with fibromyalgia; do not infer this selected participant's diagnosis from the dataset title. See dataset metadata for citation and terms.


## Setup
### 1. Connect Drive and create your workspace
Approve the Google Drive connection when Colab asks. The notebook writes inside `MyDrive/Neuroimaging-Training`. Each session gets a new result folder, so previous results remain available. Downloaded inputs are reused only after checksum verification.

Mounting Drive makes your Drive accessible to code in this Colab runtime. Run this reviewed notebook in your own account; do not share a live runtime.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import datetime, hashlib, json, os, platform, shutil, subprocess, sys, urllib.request
ROOT = Path('/content/drive/MyDrive/Neuroimaging-Training')
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
INPUT = ROOT / 'data' / 'ds004144' / 'sub-002' / 'anat'
RESULTS = ROOT / 'results' / RUN_ID
for folder in [INPUT, RESULTS, ROOT / 'notes']:
    folder.mkdir(parents=True, exist_ok=True)
WORK = Path('/content/neuroimaging-pilot') / RUN_ID
WORK.mkdir(parents=True, exist_ok=True)
print('Your results will be saved in:', RESULTS)


### 2. Install the analysis tools
This follows the official FastSurfer Colab approach, using a fixed source commit. It keeps Colab's installed PyTorch and torchvision. The Colab environment and resolved dependencies can change; they are recorded below. If installation fails or asks for a restart, restart the session and rerun from cell 1; do not continue past an error.


In [ ]:
import re, tomllib
FS_COMMIT = '1dc024735371f65bc4b94a272cf596be6ddf4f21'
FS_HOME = Path('/content/fastsurfer-training')
if not (FS_HOME / '.git').exists():
    subprocess.run(['git', 'clone', '--quiet', 'https://github.com/Deep-MI/FastSurfer.git', str(FS_HOME)], check=True)
subprocess.run(['git', '-C', str(FS_HOME), 'checkout', '--quiet', FS_COMMIT], check=True)
with (FS_HOME / 'pyproject.toml').open('rb') as handle:
    dependencies = tomllib.load(handle)['project']['dependencies']
dependencies = [d for d in dependencies if re.split(r'[\s<>=!~;\[]', d, maxsplit=1)[0] not in ('torch', 'torchvision')]
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *dependencies, 'matplotlib', 'pandas', 'ipywidgets'], check=True)
import numpy as np, nibabel as nib, pandas as pd, matplotlib.pyplot as plt, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU fallback')
(RESULTS / 'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))


## Steps
### 3. Download one scan directly from OpenNeuro's public storage
No manual ZIP upload is needed. The image comes from OpenNeuro's S3 mirror. Since the mirror path can change, we verify the bytes against the MD5 annex key in a frozen dataset commit. If it differs, this cell stops rather than accepting changed data. Metadata is fetched from that frozen commit.

Expected input: `sub-002_T1w.nii.gz`, 6,877,450 bytes. This is a selected teaching input, not a complete BIDS download. Inspect the printed licence, title, and scanner strength.


In [ ]:
DATASET_COMMIT = 'b6e525adf7b861d5dab7e539bff19f90ce482032'
RELATIVE_PATH = 'sub-002/anat/sub-002_T1w.nii.gz'
EXPECTED_MD5 = 'ca084e623f0156ef16d4d9aeff7f18b7'
EXPECTED_BYTES = 6877450
IMAGE_URL = 'https://s3.amazonaws.com/openneuro.org/ds004144/' + RELATIVE_PATH
T1 = INPUT / 'sub-002_T1w.nii.gz'
def digest(path, algorithm='sha256'):
    h = hashlib.new(algorithm)
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()
if not T1.exists():
    temporary = T1.with_name(T1.name + '.partial')
    with urllib.request.urlopen(IMAGE_URL, timeout=120) as response, temporary.open('wb') as handle:
        shutil.copyfileobj(response, handle)
    if temporary.stat().st_size != EXPECTED_BYTES or digest(temporary, 'md5') != EXPECTED_MD5:
        raise ValueError('Download differs from frozen source. Stop and report this cell; do not bypass the check.')
    temporary.rename(T1)
assert T1.stat().st_size == EXPECTED_BYTES and digest(T1, 'md5') == EXPECTED_MD5, 'Existing input failed verification.'
metadata_root = f'https://raw.githubusercontent.com/OpenNeuroDatasets/ds004144/{DATASET_COMMIT}/'
for relative, destination in [('dataset_description.json', INPUT.parents[1] / 'dataset_description.json'),
                              ('sub-002/anat/sub-002_T1w.json', INPUT / 'sub-002_T1w.json')]:
    with urllib.request.urlopen(metadata_root + relative, timeout=60) as response:
        payload = response.read()
    json.loads(payload)
    destination.write_bytes(payload)
description = json.loads((INPUT.parents[1] / 'dataset_description.json').read_text())
sidecar = json.loads((INPUT / 'sub-002_T1w.json').read_text())
provenance = dict(dataset='ds004144', dataset_commit=DATASET_COMMIT,
                  dataset_doi=description.get('DatasetDOI'), licence=description.get('License'),
                  source_url=IMAGE_URL, relative_path=RELATIVE_PATH, input_sha256=digest(T1),
                  input_md5=digest(T1, 'md5'), fastsurfer_commit=FS_COMMIT,
                  run_id=RUN_ID, python=platform.python_version(), torch=torch.__version__,
                  cuda=torch.version.cuda, qc_status='pending', method='hard-label voxel count')
(RESULTS / 'provenance.json').write_text(json.dumps(provenance, indent=2))
print(description['Name'], '\nLicence:', description.get('License'))
print('DOI:', description.get('DatasetDOI'), '\nField strength:', sidecar.get('MagneticFieldStrength'), 'T')
print('Verified input:', T1)


### 4. Look at the MRI before processing
Check that you see a full brain, plausible contrast, and no major motion. These three slices are an initial view, not complete QC. If the scan looks unsuitable, stop and record why.


In [ ]:
local_t1 = WORK / T1.name
shutil.copy2(T1, local_t1)
t1_image = nib.as_closest_canonical(nib.load(local_t1))
assert len(t1_image.shape) == 3, 'Expected one 3D anatomical volume.'
t1_data = t1_image.get_fdata(dtype=np.float32)
assert np.isfinite(t1_data).all(), 'Input has non-finite intensities.'
print('Shape:', t1_image.shape, 'Voxel size (mm):', nib.affines.voxel_sizes(t1_image.affine))
def show_input(data):
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    vmax = np.percentile(data, 99)
    for axis, (ax, title) in enumerate(zip(axes, ['Sagittal', 'Coronal', 'Axial'])):
        ax.imshow(np.take(data, data.shape[axis] // 2, axis=axis).T, origin='lower', cmap='gray', vmax=vmax)
        ax.set_title(title); ax.axis('off')
    plt.tight_layout()
    return fig
fig = show_input(t1_data)
fig.savefig(RESULTS / 'input_preview.png', dpi=150)
plt.show()


### 5. Segment the scan
This runs the quick segmentation configuration from the official Colab example: no surfaces, bias-field correction, cerebellar subsegmentation, hypothalamic subsegmentation, or corpus-callosum module. Basic cerebellar labels remain in the whole-brain segmentation.

The field-strength flag comes from the sidecar. Setup/model downloads may take longer than inference. Keep Colab connected. The log is streamed and saved directly to Drive; a failed process stops this notebook.


In [ ]:
SUBJECT = 'sub-002'
SUBJECTS = WORK / 'subjects'
command = ['bash', str(FS_HOME / 'run_fastsurfer.sh'), '--t1', str(local_t1),
           '--sd', str(SUBJECTS), '--sid', SUBJECT, '--seg_only', '--py', sys.executable,
           '--no_biasfield', '--no_cereb', '--no_hypothal', '--no_cc', '--allow_root']
if sidecar.get('MagneticFieldStrength') == 3:
    command.append('--3T')
provenance['command'] = command
(RESULTS / 'provenance.json').write_text(json.dumps(provenance, indent=2))
environment = dict(os.environ, FASTSURFER_HOME=str(FS_HOME))
with (RESULTS / 'segmentation.log').open('w') as log:
    process = subprocess.Popen(command, cwd=FS_HOME, env=environment,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end='')
        log.write(line); log.flush()
    return_code = process.wait()
if return_code:
    raise RuntimeError(f'FastSurfer failed ({return_code}). See segmentation.log; do not continue.')
OUTPUT = SUBJECTS / SUBJECT
SEG = OUTPUT / 'mri/aparc.DKTatlas+aseg.deep.mgz'
assert SEG.is_file(), 'Expected segmentation is missing.'
shutil.copytree(OUTPUT, RESULTS / SUBJECT, dirs_exist_ok=True)
print('Segmentation copied to Drive.')


## Checks
### 6. Inspect the labels over the matching MRI
Use the **conformed MRI**, which shares the segmentation's grid. Comparing the original and segmentation arrays without alignment can show a misleading overlay. The code refuses mismatched grids.

Scroll through all three planes. Check brain coverage, ventricles, deep nuclei, and cortical boundaries. A colourful output alone does not establish that segmentation is correct. Static previews are saved; sliders require a live Colab runtime.


In [ ]:
CONFORMED = OUTPUT / 'mri/orig.mgz'
if not CONFORMED.exists():
    raise FileNotFoundError('Matching conformed MRI mri/orig.mgz missing. Stop rather than overlaying the original grid.')
seg_image = nib.as_closest_canonical(nib.load(SEG))
conformed_image = nib.as_closest_canonical(nib.load(CONFORMED))
assert seg_image.shape == conformed_image.shape and np.allclose(seg_image.affine, conformed_image.affine), 'QC grid mismatch.'
seg = np.asarray(seg_image.dataobj)
assert np.isfinite(seg).all() and np.all(seg == np.rint(seg)), 'Expected finite integer labels.'
seg = seg.astype(np.int32)
anatomy = conformed_image.get_fdata(dtype=np.float32)
assert (seg > 0).any(), 'Segmentation is empty.'
# Consecutive colour indices avoid assigning near-identical colours to large label IDs.
label_ids = np.unique(seg)
colour_index = np.searchsorted(label_ids, seg)
def overlay(axis=2, fraction=0.5):
    index = min(int(fraction * (seg.shape[axis] - 1)), seg.shape[axis] - 1)
    base = np.take(anatomy, index, axis=axis).T
    labels = np.take(colour_index, index, axis=axis).T
    mask = np.take(seg, index, axis=axis).T == 0
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(base, origin='lower', cmap='gray', vmax=np.percentile(anatomy, 99))
    ax.imshow(np.ma.masked_where(mask, labels), origin='lower', cmap='turbo',
              vmin=0, vmax=len(label_ids), alpha=0.35, interpolation='nearest')
    ax.set_title(f'{["Sagittal", "Coronal", "Axial"][axis]} slice {index}'); ax.axis('off')
    return fig
for axis in range(3):
    fig = overlay(axis)
    fig.savefig(RESULTS / f'qc_plane_{axis}.png', dpi=150)
    plt.show()
import ipywidgets as widgets
@widgets.interact(axis=widgets.Dropdown(options=[('Sagittal', 0), ('Coronal', 1), ('Axial', 2)], value=2),
                  fraction=widgets.FloatSlider(min=0, max=1, step=0.01, value=0.5))
def explore(axis, fraction):
    overlay(axis, fraction); plt.show()


### 7. Calculate regional volumes
For each non-background label: **voxel count × voxel volume (mm³)**; divide by 1,000 for mL. Names come from the lookup table shipped with the pinned FastSurfer source.

These are **hard-label volumes**, not partial-volume-corrected FreeSurfer/FastSurfer statistics, cortical thickness, or eTIV-normalised volumes. Do not mix these measurement types in a study. No inferential statistics are justified by one participant.


In [ ]:
def regional_table(label_data, affine, lookup):
    ids, counts = np.unique(label_data, return_counts=True)
    voxel_mm3 = float(abs(np.linalg.det(affine[:3, :3])))
    rows = [dict(label_id=int(i), region=lookup.get(int(i), f'UNMAPPED_{i}'),
                 voxel_count=int(n), volume_mm3=float(n * voxel_mm3),
                 volume_ml=float(n * voxel_mm3 / 1000))
            for i, n in zip(ids, counts) if i != 0]
    table = pd.DataFrame(rows).sort_values('label_id').reset_index(drop=True)
    assert np.isclose(table.volume_mm3.sum(), np.count_nonzero(label_data) * voxel_mm3)
    return table
lookup = {}
lut_path = FS_HOME / 'FastSurferCNN/config/FreeSurferColorLUT.txt'
for line in lut_path.read_text().splitlines():
    fields = line.split()
    if fields and fields[0].isdigit() and len(fields) >= 2:
        lookup[int(fields[0])] = fields[1]
table = regional_table(seg, seg_image.affine, lookup)
if table.region.str.startswith('UNMAPPED_').any():
    raise ValueError('Unmapped labels found. Review the lookup table before exporting.')
table.insert(0, 'participant_id', SUBJECT)
table.to_csv(RESULTS / 'regional_volumes.csv', index=False)
display(table.head(15))
print('Regions exported:', len(table))
# Display selected paired structures; a missing label stays missing, never silently becomes zero.
selected = table[table.label_id.isin([10, 49, 11, 50, 12, 51, 17, 53, 18, 54])].copy()
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(selected.region, selected.volume_ml, color='#287e8d')
ax.set_xlabel('Hard-label volume (mL)'); ax.set_title('One participant: selected regional volumes')
fig.tight_layout(); fig.savefig(RESULTS / 'selected_volumes.png', dpi=150); plt.show()


### 8. Record your QC decision and finish
Edit the form, then run. Use **needs_review** if uncertain; supervisor review is a separate checkpoint. Do not mark a scan as passed from only three screenshots. Your CSV is a preliminary measurement until QC is complete.


In [ ]:
QC_STATUS = "needs_review" #@param ["needs_review", "pass", "fail"]
QC_NOTES = "" #@param {type:"string"}
if QC_STATUS in ('pass', 'fail') and not QC_NOTES.strip():
    raise ValueError('Describe what you inspected and why you chose pass/fail.')
provenance.update(qc_status=QC_STATUS, qc_notes=QC_NOTES,
                  segmentation_sha256=digest(SEG), lookup_sha256=digest(lut_path),
                  volume_table_sha256=digest(RESULTS / 'regional_volumes.csv'),
                  completed_utc=datetime.datetime.now(datetime.timezone.utc).isoformat())
(RESULTS / 'provenance.json').write_text(json.dumps(provenance, indent=2))
summary = f"""# Pilot handover
Run: {RUN_ID}
Dataset: ds004144; input: {RELATIVE_PATH}
QC: {QC_STATUS}
Notes: {QC_NOTES}
Measure: hard-label voxel counts in mm3 and mL, not partial-volume-corrected statistics.
This single scan cannot establish disease differences.
Next: explain one region's volume calculation and request supervisor review.
"""
(RESULTS / 'handover.md').write_text(summary)
print('Saved:', RESULTS)
print('\n'.join(p.name for p in RESULTS.iterdir()))


## Next steps
1. Open Drive → Neuroimaging-Training → results → this run's folder. Confirm the CSV, provenance, environment, logs, images, and segmentation folder exist.
2. Save this executed notebook copy in Drive. If disconnected, reconnect and rerun setup; completed outputs remain in Drive.
3. Add a short progress comment to your assigned GitHub task: stage reached, QC outcome, obstacle, next action. Keep the full imaging outputs in Drive.
4. Explain how 4,000 voxels of 1 mm³ produce 4,000 mm³ = 4 mL. These numbers are a teaching example, not this scan's result.
5. Have the supervisor inspect the pilot before adding a second scan or planning group comparisons. A later study needs a defined cohort, covariates, a consistent head-size measure, missing-data handling, and multiple-comparison planning.

### Troubleshooting
- **Drive connection fails:** check the Google account and rerun cell 1.
- **No GPU:** try another session later or use CPU; processing will be slower.
- **Checksum/download error:** stop and report the exact cell. Do not disable verification.
- **Import/package error:** save the message and environment details; Colab packages change.
- **Segmentation error:** read `segmentation.log`; do not interpret partial outputs.
- **Disconnected before copy:** runtime files may be lost; run the pilot again. Drive inputs/results already copied remain.

### Sources and method
- [Dataset metadata at the frozen commit](https://github.com/OpenNeuroDatasets/ds004144/blob/b6e525adf7b861d5dab7e539bff19f90ce482032/dataset_description.json) (CC0; follow its acknowledgement instructions).
- [Official FastSurfer quick Colab example](https://github.com/Deep-MI/FastSurfer/blob/1dc024735371f65bc4b94a272cf596be6ddf4f21/Tutorial/Tutorial_FastSurferCNN_QuickSeg.ipynb): source for the installation approach and quick-segmentation configuration; FastSurfer retains its own licence.
- [FastSurfer documentation](https://deep-mi.org/FastSurfer/stable/overview/QUICKSTART.html).
- [NiBabel spatial images](https://nipy.org/nibabel/coordinate_systems.html): affine geometry used for alignment and voxel volume.

New teaching wrapper and calculation code for Donggyurim's placement programme. No results are prefilled. Local checks cover code syntax and synthetic volume arithmetic; Drive authentication, actual downloads, model inference, and visual QC require the first Colab pilot.
